# 01 — Exploración del dataset YT-MR
Distribución de la curva Most Replayed, duración de candidatos y correlación de cada feature con la etiqueta. Requiere haber corrido `run_all.py --only features`.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.common.config import load_config
from src.data.featurize import load_table
cfg = load_config()
cand, bins = load_table(cfg, 'sentence'), load_table(cfg, 'bin')
print(cand.video_id.nunique(), 'videos,', len(cand), 'candidatos')

In [ ]:
# curva promedio de Most Replayed por posición relativa (sesgo de inicio/fin)
bins['pos'] = bins.groupby('video_id').cumcount()
bins.groupby('pos')['y_mr'].mean().plot(figsize=(8,3), title='Most Replayed medio por bin (1% del video)'); plt.xlabel('bin'); plt.show()

In [ ]:
# duración de candidatos y proporción de relevantes
cand['f_duration'].hist(bins=30, figsize=(6,3)); plt.title('Duración de candidatos (s)'); plt.show()
print('relevantes:', cand.y_relevant.mean().round(3))

In [ ]:
# correlación de Spearman de cada feature con y_mr (dentro de video, promediada)
feats = [c for c in cand.columns if c[:2] in ('k_','s_','a_','h_','f_')]
corr = cand.groupby('video_id').apply(lambda g: g[feats].corrwith(g.y_mr, method='spearman')).mean().sort_values()
corr.plot.barh(figsize=(6,12), title='Spearman medio con Most Replayed'); plt.show()

In [ ]:
# por idioma
cand.groupby('lang').agg(videos=('video_id','nunique'), candidatos=('win_id','size'), relevantes=('y_relevant','mean'))